In [26]:
# Import packages
import numpy as np
import matplotlib.pyplot as plt
from MDPSetup import *
from ValueIteration import *

# 2-echelon problem with lead times at DC only

## Centralised problem

In [27]:
# Problem parameters
capacity = (-400, 400)
increment = 100
lead_times = [1, 0]
n_ech = 2
demand_dist = {0: 0.2, 100: 0.6, 200: 0.2}
max_demand = max(demand_dist.keys())
hold_costs = [1, 0.5]
backlog_costs = [50, 25]

# Perform value iteration
gamma = 0.999
S, state_idx = create_state_space(capacity, increment, n_ech, lead_times)
A, action_idx = create_action_space(capacity, increment, n_ech, max_demand)
P = create_P(S, A, state_idx, action_idx, demand_dist, capacity, lead_times)
R = create_R(S, A, state_idx, action_idx, demand_dist, hold_costs, backlog_costs, lead_times)
bellman_eq_cL = cL_value_update_func(state_idx, action_idx, capacity, demand_dist)
V_init = dict([(s, 0) for s in S])


In [28]:

cL_results = value_iteration(S, A, P, R, gamma, max_iterations = 100,
                                 bellman_eq=bellman_eq_cL, V_init = V_init, theta = 1e-7)
cL_optimal_policy = cL_results["optimal_policy"]
cL_cost_function = cL_results["value_function"]

In [29]:
cL_optimal_policy

{(-400, -400, 0): (200, 600),
 (-400, -400, 100): (200, 600),
 (-400, -400, 200): (100, 500),
 (-400, -400, 300): (0, 400),
 (-400, -400, 400): (0, 400),
 (-400, -400, 500): (0, 400),
 (-400, -400, 600): (0, 400),
 (-400, -400, 700): (0, 400),
 (-400, -400, 800): (0, 400),
 (-400, -300, 0): (200, 500),
 (-400, -300, 100): (200, 500),
 (-400, -300, 200): (200, 500),
 (-400, -300, 300): (100, 400),
 (-400, -300, 400): (0, 300),
 (-400, -300, 500): (0, 300),
 (-400, -300, 600): (0, 300),
 (-400, -300, 700): (0, 300),
 (-400, -300, 800): (0, 300),
 (-400, -200, 0): (200, 400),
 (-400, -200, 100): (200, 400),
 (-400, -200, 200): (200, 400),
 (-400, -200, 300): (200, 400),
 (-400, -200, 400): (100, 300),
 (-400, -200, 500): (0, 200),
 (-400, -200, 600): (0, 200),
 (-400, -200, 700): (100, 300),
 (-400, -200, 800): (0, 200),
 (-400, -100, 0): (300, 300),
 (-400, -100, 100): (300, 300),
 (-400, -100, 200): (200, 300),
 (-400, -100, 300): (200, 300),
 (-400, -100, 400): (200, 300),
 (-400, -100

# Plotting Code
## Plotting functions

In [ ]:
def calculate_ip(state, lead_times, n_ech):
    ips = []
    next_lead_idx = n_ech
    for ech in range(n_ech):
        ips.append(state[ech] + sum(state[next_lead_idx:next_lead_idx+lead_times[ech]]))
        next_lead_idx += lead_times[ech]

    return tuple(ips)

True

In [ ]:


def make_policy_plot_dict(optimal_dict: Dict, system_type: str ="Centralised", lead_times: List, n_ech: int):
    IPs = set(calculate_ip(state, lead_times, n_ech) for state in optimal_dict) 
    
    DC_pol = dict((x1, [[], []]) for (x1, x2) in optimal_dict)  # Storing W IP and W order quantity for each DC IP (key)

    if system_type == "centralised":
        W_pol = dict((x2, [[], []]) for (x1, x2) in optimal_dict)   # Storing DC IP and DC order quantity for each W IP (key)

        for state, (q_dc, q_w) in optimal_dict.items():
            ip_dc, ip_w = calculate_ip(state, lead_times, n_ech)
            DC_pol[ip_dc][0].append(ip_w)
            DC_pol[ip_dc][1].append(q_w)
            W_pol[ip_w][0].append(ip_dc)
            W_pol[ip_w][1].append(q_dc)

        return DC_pol, W_pol
    
    # else: # COMPLETE FOR DECENTRALISED SYSTEM



def generate_policy_plot(optimal_dict: Dict, capacity: tuple, 
                         lead_times: List, n_ech: int, system_type: str = "Centralised", 
                         colour_by: str = "DC", bbox_to_anchor=None, savefig=False):
    if system_type == "Centralised":
        DC_pol, W_pol = make_policy_plot_dict(optimal_dict, system_type, lead_times, n_ech)
    else:
        DC_pol = make_policy_plot_dict(optimal_dict, system_type, lead_times, n_ech)

    
    plot_pol = DC_pol if colour_by == "DC" else W_pol
    x_site = "DC" if colour_by == "W" else "warehouse"


    